# 📚 Student Performance Analysis and Prediction

**Goal:** Understand what affects student marks, explore the data, and build a simple model to predict marks.

**Topics covered:**
- Loading and cleaning data
- Exploratory Data Analysis (EDA)
- Visualizations: Scatter plot, Bar chart, Heatmap
- Linear Regression model
- Predicting marks from study hours

---

## Step 1: Import Libraries

We first import all the tools (libraries) we need.
- **pandas** → for handling data (like Excel in Python)
- **numpy** → for math operations
- **matplotlib / seaborn** → for making charts
- **sklearn** → for machine learning (our prediction model)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# This makes plots show up inside the notebook
%matplotlib inline

print("All libraries imported successfully!")

## Step 2: Load the Dataset

We load the CSV file that contains student information.
Make sure `student_data.csv` is in the same folder as this notebook.

In [ ]:
# Load the CSV file into a DataFrame (like a table)
df = pd.read_csv("student_data.csv")

# Show the first 5 rows to see what the data looks like
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# How many rows and columns do we have?
print(f"Shape of dataset: {df.shape}")
print(f"Total students: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")

print("\nColumn names:")
print(df.columns.tolist())

## Step 3: Clean the Data

Real-world data often has missing values. We need to handle them before analysis.

**Strategy:** Fill missing numbers with the average (mean) of that column.

In [ ]:
# Check how many missing values are in each column
print("Missing values in each column:")
print(df.isnull().sum())

In [ ]:
# Fill missing values with column mean (average)
df["StudyHours"] = df["StudyHours"].fillna(df["StudyHours"].mean())
df["Attendance"] = df["Attendance"].fillna(df["Attendance"].mean())
df["Marks"]      = df["Marks"].fillna(df["Marks"].mean())

# Check again — should all be 0 now
print("Missing values after cleaning:")
print(df.isnull().sum())
print("\nData is clean! ✅")

## Step 4: Exploratory Data Analysis (EDA)

EDA means exploring the data to understand it better before doing any predictions.
We look at basic statistics, distributions, and patterns.

In [ ]:
# Basic statistics: mean, min, max, etc.
print("Basic Statistics:")
df[["StudyHours", "Attendance", "Marks"]].describe().round(2)

In [ ]:
# How many boys and girls are in the dataset?
print("Gender distribution:")
print(df["Gender"].value_counts())

print("\nGrade distribution:")
print(df["Grade"].value_counts().sort_index())

In [ ]:
# Average marks for each grade
print("Average marks per grade:")
df.groupby("Grade")["Marks"].mean().round(1).sort_index()

## Step 5: Visualization 1 — Scatter Plot (Study Hours vs Marks)

A scatter plot shows us if there's a relationship between two things.
Here we check: **Does studying more → higher marks?**

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(df["StudyHours"], df["Marks"], color="steelblue", alpha=0.7, edgecolors="white", s=80)

plt.xlabel("Study Hours (per day)")
plt.ylabel("Marks (out of 100)")
plt.title("Study Hours vs Marks")
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print("Observation: Students who study more tend to score higher marks.")

## Step 6: Visualization 2 — Bar Chart (Attendance vs Marks)

We group students by their attendance range and see the average marks for each group.
This helps answer: **Does attending more classes help students score better?**

In [ ]:
# Create attendance buckets (groups)
df["AttendanceBucket"] = pd.cut(
    df["Attendance"],
    bins=[50, 70, 80, 90, 100],
    labels=["50–70%", "70–80%", "80–90%", "90–100%"]
)

# Average marks for each attendance group
bucket_avg = df.groupby("AttendanceBucket", observed=True)["Marks"].mean().reset_index()

# Draw the bar chart
plt.figure(figsize=(8, 5))
colors = ["#f28b82", "#fbbc04", "#34a853", "#4285f4"]

plt.bar(bucket_avg["AttendanceBucket"].astype(str), bucket_avg["Marks"], color=colors, edgecolor="white", width=0.5)

# Add value labels on top of each bar
for i, val in enumerate(bucket_avg["Marks"]):
    plt.text(i, val + 0.5, f"{val:.1f}", ha="center", fontsize=11, fontweight="bold")

plt.xlabel("Attendance Range")
plt.ylabel("Average Marks")
plt.title("Attendance Range vs Average Marks")
plt.ylim(0, 100)
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print("Observation: Higher attendance is linked to higher average marks.")

## Step 7: Visualization 3 — Heatmap (Correlation)

A **correlation heatmap** shows how strongly two variables are related.
- Value close to **1** → strong positive relationship
- Value close to **0** → no relationship
- Value close to **-1** → opposite relationship

In [ ]:
# Calculate correlation between numeric columns
corr_matrix = df[["StudyHours", "Attendance", "Marks"]].corr()

print("Correlation Matrix:")
print(corr_matrix.round(2))

# Draw the heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, cmap="YlGnBu", fmt=".2f", linewidths=0.5, square=True)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

print("\nObservation: StudyHours and Attendance both have a strong positive correlation with Marks.")

## Step 8: Build a Linear Regression Model

**Linear Regression** is a simple machine learning technique.
It finds the best straight line through the data to make predictions.

We use:
- **Input (X):** Study Hours
- **Output (y):** Marks

We split data into **training set** (80%) and **test set** (20%).

In [ ]:
# Define input and output
X = df[["StudyHours"]]  # Feature (input)
y = df["Marks"]         # Target (output)

# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

In [ ]:
# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully! ✅")
print(f"\nModel Equation:")
print(f"  Marks = {model.coef_[0]:.2f} × StudyHours + {model.intercept_:.2f}")
print(f"\nThis means: for every extra hour of study, marks increase by ~{model.coef_[0]:.1f} points.")

## Step 9: Test the Model (Evaluate Performance)

After training, we check how well the model predicts on data it hasn't seen.
- **R² Score** → How much of the variation in marks is explained by study hours (higher = better, max is 1.0)
- **RMSE** → Average error in predictions (lower = better)

In [ ]:
# Predict on the test data
y_pred = model.predict(X_test)

# Calculate performance metrics
r2  = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score : {r2:.2f}  (closer to 1.0 is better)")
print(f"RMSE     : {rmse:.2f} (average prediction error in marks)")

# Show actual vs predicted side by side
results = pd.DataFrame({
    "Actual Marks":    y_test.values,
    "Predicted Marks": y_pred.round(1)
})
print("\nSample predictions:")
results.head(10)

## Step 10: Visualize the Regression Line

In [ ]:
plt.figure(figsize=(8, 5))

# All data points
plt.scatter(df["StudyHours"], df["Marks"], color="steelblue", alpha=0.6, label="Actual Data", edgecolors="white")

# Regression line
x_line = np.linspace(0, 10, 100).reshape(-1, 1)
y_line = model.predict(x_line)
plt.plot(x_line, np.clip(y_line, 0, 100), color="red", linewidth=2, linestyle="--", label="Regression Line")

plt.xlabel("Study Hours (per day)")
plt.ylabel("Marks (out of 100)")
plt.title("Linear Regression: Study Hours vs Marks")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

## Step 11: Make Your Own Prediction

Now the fun part — let's use our trained model to predict marks for any number of study hours!

In [ ]:
# ---- Change this value to predict for different study hours ----
my_study_hours = 5.0  # hours per day
# ---------------------------------------------------------------

predicted_marks = model.predict([[my_study_hours]])[0]
predicted_marks = max(0, min(100, round(predicted_marks, 1)))  # keep within 0-100

print(f"If a student studies {my_study_hours} hours per day...")
print(f"Predicted Marks: {predicted_marks} / 100")

# Grade label
if predicted_marks >= 80:
    grade = "A (Excellent)"
elif predicted_marks >= 60:
    grade = "B (Good)"
elif predicted_marks >= 45:
    grade = "C (Average)"
else:
    grade = "F (Needs Improvement)"

print(f"Estimated Grade: {grade}")

In [ ]:
# Quick predictions table for 1 to 10 hours
print("Predictions for different study hours:\n")
print(f"{'Study Hours':<15} {'Predicted Marks':<18} {'Grade'}")
print("-" * 45)

for h in range(1, 11):
    m = max(0, min(100, round(model.predict([[h]])[0], 1)))
    if m >= 80:   g = "A"
    elif m >= 60: g = "B"
    elif m >= 45: g = "C"
    else:         g = "F"
    print(f"{h:<15} {m:<18} {g}")

---
## ✅ Summary

Here's what we did in this project:

| Step | Task |
|------|------|
| 1 | Imported libraries |
| 2 | Loaded the student dataset |
| 3 | Cleaned missing values |
| 4 | Explored the data (EDA) |
| 5 | Scatter plot: Study Hours vs Marks |
| 6 | Bar chart: Attendance vs Marks |
| 7 | Heatmap: Correlation between features |
| 8 | Built a Linear Regression model |
| 9 | Evaluated the model (R², RMSE) |
| 10 | Visualized the regression line |
| 11 | Made predictions |

**Key Insights:**
- Students who study more hours tend to score higher marks.
- Regular attendance also positively impacts performance.
- Linear Regression gives us a simple formula to predict marks from study hours.

---
*Next steps: Try adding more features (like attendance) to improve predictions!*